# IMDb Sentiment Analysis ML Pipeline
This notebook contains the complete pipeline for sentiment analysis using the IMDb dataset.

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Download stopwords
nltk.download('stopwords')

## 1. Load Data

In [ ]:
df = pd.read_csv('IMDB Dataset.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Text Preprocessing

In [ ]:
def clean_text(text):
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    # Convert to lowercase
    text = text.lower()
    # Remove extra whitespaces
    text = ' '.join(text.split())
    return text

df['review'] = df['review'].apply(clean_text)
df['sentiment'] = df['sentiment'].apply(lambda x: 1 if x == 'positive' else 0)
df.head()

## 3. Train-Test Split

In [ ]:
X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, stop_words=stopwords.words('english'))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## 5. Model Training

In [ ]:
model = LogisticRegression()
model.fit(X_train_tfidf, y_train)

## 6. Evaluation

In [ ]:
y_pred = model.predict(X_test_tfidf)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

## 7. Save Model and Vectorizer

In [ ]:
joblib.dump(model, 'sentiment_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("Model and Vectorizer saved successfully!")